In [1]:
from __future__ import annotations

import json
import re
from pathlib import Path

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix, hstack

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score,
)
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)


# ============================================================
# CONFIGURATION
# ============================================================

CURRENT_DIRECTORY = Path.cwd()

PROJECT_ROOT = (
    CURRENT_DIRECTORY.parent
    if CURRENT_DIRECTORY.name.lower() == "notebooks"
    else CURRENT_DIRECTORY
)

MODEL_INPUT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "amazon_multimodal"
    / "model_input"
)

PRODUCTION_MODEL_ROOT = (
    PROJECT_ROOT
    / "models"
    / "price_prediction"
    / "production_v1"
)

REPORT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "reports"
    / "price_prediction"
    / "production_v1"
)

PRODUCTION_MODEL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

REPORT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

MODEL_FILE = (
    PRODUCTION_MODEL_ROOT
    / "price_model.joblib"
)

PREPROCESSOR_FILE = (
    PRODUCTION_MODEL_ROOT
    / "structured_preprocessor.joblib"
)

TFIDF_FILE = (
    PRODUCTION_MODEL_ROOT
    / "title_tfidf.joblib"
)

FEATURE_CONFIG_FILE = (
    PRODUCTION_MODEL_ROOT
    / "feature_config.json"
)

MODEL_METADATA_FILE = (
    PRODUCTION_MODEL_ROOT
    / "model_metadata.json"
)

RANDOM_STATE = 42


# ============================================================
# LOAD DATA
# ============================================================

train_df = pd.read_parquet(
    MODEL_INPUT_ROOT / "train.parquet"
).reset_index(drop=True)

validation_df = pd.read_parquet(
    MODEL_INPUT_ROOT / "validation.parquet"
).reset_index(drop=True)

test_df = pd.read_parquet(
    MODEL_INPUT_ROOT / "test.parquet"
).reset_index(drop=True)

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)


# ============================================================
# FEATURE ENGINEERING
# ============================================================

def extract_first_number(text: str) -> float:

    numbers = re.findall(
        r"\d+(?:\.\d+)?",
        str(text),
    )

    if not numbers:
        return 0.0

    try:
        return float(numbers[0])
    except Exception:
        return 0.0


def create_features(
    dataframe: pd.DataFrame,
) -> pd.DataFrame:

    df = dataframe.copy()

    df["title"] = (
        df["title"]
        .fillna("")
        .astype(str)
    )

    df["category_name"] = (
        df["category_name"]
        .fillna("Unknown")
        .astype(str)
    )

    df["stars"] = (
        pd.to_numeric(
            df["stars"],
            errors="coerce",
        )
        .fillna(0)
    )

    reviews = (
        pd.to_numeric(
            df["reviews"],
            errors="coerce",
        )
        .fillna(0)
    )

    bought = (
        pd.to_numeric(
            df["boughtInLastMonth"],
            errors="coerce",
        )
        .fillna(0)
    )

    df["reviews_log1p"] = np.log1p(
        reviews
    )

    df["bought_log1p"] = np.log1p(
        bought
    )

    df["isBestSeller"] = (
        df["isBestSeller"]
        .fillna(False)
        .astype(int)
    )

    df["cluster_id"] = (
        pd.to_numeric(
            df["cluster_id"],
            errors="coerce",
        )
        .fillna(-1)
    )

    # --------------------------------------------------------
    # TITLE STATISTICS
    # --------------------------------------------------------

    df["title_char_length"] = (
        df["title"].str.len()
    )

    df["title_word_count"] = (
        df["title"]
        .str.split()
        .str.len()
        .fillna(0)
    )

    df["title_digit_count"] = (
        df["title"]
        .str.count(r"\d")
    )

    df["title_uppercase_count"] = (
        df["title"].apply(
            lambda text:
            sum(
                character.isupper()
                for character in text
            )
        )
    )

    df["title_first_number"] = (
        df["title"]
        .apply(
            extract_first_number
        )
    )

    lower_title = (
        df["title"]
        .str.lower()
    )

    patterns = {

        "has_gb":
            r"\b\d+(?:\.\d+)?\s*gb\b",

        "has_tb":
            r"\b\d+(?:\.\d+)?\s*tb\b",

        "has_ram":
            r"\b(?:ram|memory)\b",

        "has_inch":
            r'\b\d+(?:\.\d+)?\s*(?:inch|inches|")',

        "has_cm":
            r"\b\d+(?:\.\d+)?\s*cm\b",

        "has_kg":
            r"\b\d+(?:\.\d+)?\s*kg\b",

        "has_gram":
            r"\b\d+(?:\.\d+)?\s*(?:g|gram|grams)\b",

        "has_watt":
            r"\b\d+(?:\.\d+)?\s*(?:w|watt|watts)\b",

        "has_volt":
            r"\b\d+(?:\.\d+)?\s*(?:v|volt|volts)\b",

        "has_pack":
            r"\b(?:pack|set|pair|bundle)\b",

        "has_multipack_number":
            r"\b\d+\s*[- ]?(?:pack|piece|pcs|count|ct)\b",

        "has_pro":
            r"\bpro\b",

        "has_max":
            r"\bmax\b",

        "has_premium":
            r"\bpremium\b",

        "has_professional":
            r"\bprofessional\b",

        "has_wireless":
            r"\bwireless\b",

        "has_smart":
            r"\bsmart\b",
    }

    for feature_name, pattern in (
        patterns.items()
    ):

        df[feature_name] = (
            lower_title
            .str.contains(
                pattern,
                regex=True,
            )
            .astype(int)
        )

    return df


train_df = create_features(
    train_df
)

validation_df = create_features(
    validation_df
)

test_df = create_features(
    test_df
)


# ============================================================
# FINAL CLEAN FEATURE CONTRACT
# ============================================================

NUMERIC_FEATURES = [

    "stars",
    "reviews_log1p",
    "bought_log1p",
    "isBestSeller",

    "title_char_length",
    "title_word_count",
    "title_digit_count",
    "title_uppercase_count",
    "title_first_number",

    "has_gb",
    "has_tb",
    "has_ram",

    "has_inch",
    "has_cm",

    "has_kg",
    "has_gram",

    "has_watt",
    "has_volt",

    "has_pack",
    "has_multipack_number",

    "has_pro",
    "has_max",
    "has_premium",
    "has_professional",
    "has_wireless",
    "has_smart",

    "cluster_id",
]

CATEGORICAL_FEATURES = [
    "category_name",
]


# ============================================================
# LEAKAGE SAFETY CHECK
# ============================================================

FORBIDDEN_MODEL_FEATURES = {
    "price",
    "price_band",
    "price_log1p",
    "discount_amount",
    "discount_percentage",
    "listPrice",
}

used_features = set(
    NUMERIC_FEATURES
    + CATEGORICAL_FEATURES
)

leakage_found = (
    used_features
    & FORBIDDEN_MODEL_FEATURES
)

if leakage_found:

    raise RuntimeError(
        f"Target leakage detected: "
        f"{sorted(leakage_found)}"
    )

print(
    "✅ No target-derived features "
    "are used."
)


# ============================================================
# STRUCTURED PREPROCESSOR
# ============================================================

structured_preprocessor = (
    ColumnTransformer(
        transformers=[
            (
                "numeric",
                StandardScaler(),
                NUMERIC_FEATURES,
            ),

            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                ),
                CATEGORICAL_FEATURES,
            ),
        ]
    )
)


X_train_structured = (
    structured_preprocessor
    .fit_transform(train_df)
)

X_validation_structured = (
    structured_preprocessor
    .transform(validation_df)
)

X_test_structured = (
    structured_preprocessor
    .transform(test_df)
)

print(
    "Structured:",
    X_train_structured.shape
)


# ============================================================
# TF-IDF
# ============================================================

tfidf = TfidfVectorizer(

    lowercase=True,

    strip_accents="unicode",

    ngram_range=(1, 2),

    min_df=3,

    max_df=0.98,

    max_features=8000,

    sublinear_tf=True,

    dtype=np.float32,
)


X_train_tfidf = (
    tfidf.fit_transform(
        train_df["title"]
    )
)

X_validation_tfidf = (
    tfidf.transform(
        validation_df["title"]
    )
)

X_test_tfidf = (
    tfidf.transform(
        test_df["title"]
    )
)

print(
    "TF-IDF:",
    X_train_tfidf.shape
)


# ============================================================
# FINAL TRAINING MATRICES
# ============================================================

X_train = hstack(
    [
        csr_matrix(
            X_train_structured
        ),
        X_train_tfidf,
    ],
    format="csr",
)

X_validation = hstack(
    [
        csr_matrix(
            X_validation_structured
        ),
        X_validation_tfidf,
    ],
    format="csr",
)

X_test = hstack(
    [
        csr_matrix(
            X_test_structured
        ),
        X_test_tfidf,
    ],
    format="csr",
)

print(
    "Final training matrix:",
    X_train.shape
)


# ============================================================
# TARGET
# ============================================================

y_train = (
    train_df["price"]
    .astype(np.float32)
    .to_numpy()
)

y_validation = (
    validation_df["price"]
    .astype(np.float32)
    .to_numpy()
)

y_test = (
    test_df["price"]
    .astype(np.float32)
    .to_numpy()
)

y_train_log = np.log1p(
    y_train
)

y_validation_log = np.log1p(
    y_validation
)


# ============================================================
# TRAIN FINAL V1 MODEL
# ============================================================

print()
print("=" * 80)
print("TRAINING PRODUCTION V1 MODEL")
print("=" * 80)

model = lgb.LGBMRegressor(

    objective="regression",

    n_estimators=4000,

    learning_rate=0.025,

    num_leaves=63,

    max_depth=-1,

    min_child_samples=20,

    subsample=0.85,

    colsample_bytree=0.85,

    reg_alpha=0.05,

    reg_lambda=1.0,

    random_state=RANDOM_STATE,

    n_jobs=-1,

    verbosity=-1,
)


model.fit(

    X_train,

    y_train_log,

    eval_set=[
        (
            X_validation,
            y_validation_log,
        )
    ],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=150,
            verbose=False,
        )
    ],
)


# ============================================================
# EVALUATE
# ============================================================

def predict_real_price(
    X,
):

    log_prediction = (
        model.predict(X)
    )

    prediction = np.expm1(
        log_prediction
    )

    return np.clip(
        prediction,
        0,
        None,
    )


validation_prediction = (
    predict_real_price(
        X_validation
    )
)

test_prediction = (
    predict_real_price(
        X_test
    )
)


def metrics(
    actual,
    predicted,
):

    return {

        "mae":
            mean_absolute_error(
                actual,
                predicted,
            ),

        "rmse":
            np.sqrt(
                mean_squared_error(
                    actual,
                    predicted,
                )
            ),

        "median_absolute_error":
            median_absolute_error(
                actual,
                predicted,
            ),

        "r2":
            r2_score(
                actual,
                predicted,
            ),
    }


validation_metrics = metrics(
    y_validation,
    validation_prediction,
)

test_metrics = metrics(
    y_test,
    test_prediction,
)


print("\nVALIDATION")

for key, value in (
    validation_metrics.items()
):
    print(
        f"{key}: {value:.4f}"
    )


print("\nTEST")

for key, value in (
    test_metrics.items()
):
    print(
        f"{key}: {value:.4f}"
    )


# ============================================================
# SAVE PRODUCTION ARTIFACTS
# ============================================================

joblib.dump(
    model,
    MODEL_FILE,
)

joblib.dump(
    structured_preprocessor,
    PREPROCESSOR_FILE,
)

joblib.dump(
    tfidf,
    TFIDF_FILE,
)


feature_config = {

    "numeric_features":
        NUMERIC_FEATURES,

    "categorical_features":
        CATEGORICAL_FEATURES,

    "target":
        "price",

    "training_target":
        "log1p(price)",

    "required_raw_inputs": [
        "title",
        "category_name",
        "stars",
        "reviews",
        "boughtInLastMonth",
        "isBestSeller",
        "cluster_id",
    ],
}


with FEATURE_CONFIG_FILE.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        feature_config,
        file,
        indent=2,
    )


model_metadata = {

    "model_name":
        "Amazon Product Price Predictor",

    "version":
        "production_v1",

    "algorithm":
        "LightGBM",

    "text_representation":
        "TF-IDF",

    "target_transform":
        "log1p",

    "validation_metrics": {
        key: float(value)
        for key, value
        in validation_metrics.items()
    },

    "test_metrics": {
        key: float(value)
        for key, value
        in test_metrics.items()
    },

    "known_limitations": [
        (
            "Performance is weaker for "
            "luxury/high-price products."
        ),

        (
            "cluster_id currently must be "
            "generated before price inference."
        ),
    ],
}


with MODEL_METADATA_FILE.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        model_metadata,
        file,
        indent=2,
    )


print()
print("=" * 80)
print("PRODUCTION MODEL SAVED")
print("=" * 80)

print(MODEL_FILE)
print(PREPROCESSOR_FILE)
print(TFIDF_FILE)
print(FEATURE_CONFIG_FILE)
print(MODEL_METADATA_FILE)

Train: (13984, 26)
Validation: (2997, 26)
Test: (2997, 26)
✅ No target-derived features are used.
Structured: (13984, 267)
TF-IDF: (13984, 8000)
Final training matrix: (13984, 8267)

TRAINING PRODUCTION V1 MODEL


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)



VALIDATION
mae: 22.1119
rmse: 85.4636
median_absolute_error: 7.9718
r2: 0.4128

TEST
mae: 21.7364
rmse: 65.3222
median_absolute_error: 7.9332
r2: 0.5250

PRODUCTION MODEL SAVED
/Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/models/price_prediction/production_v1/price_model.joblib
/Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/models/price_prediction/production_v1/structured_preprocessor.joblib
/Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/models/price_prediction/production_v1/title_tfidf.joblib
/Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/models/price_prediction/production_v1/feature_config.json
/Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/models/price_prediction/production_v1/model_metadata.json


In [2]:
# ============================================================
# PRODUCTION INFERENCE TEST
# ============================================================

import joblib
import numpy as np
import pandas as pd

from scipy.sparse import (
    csr_matrix,
    hstack,
)


# ============================================================
# LOAD SAVED ARTIFACTS
# ============================================================

production_model = joblib.load(
    MODEL_FILE
)

production_preprocessor = joblib.load(
    PREPROCESSOR_FILE
)

production_tfidf = joblib.load(
    TFIDF_FILE
)


# ============================================================
# INFERENCE FUNCTION
# ============================================================

def predict_product_price(
    *,
    title: str,
    category_name: str,
    stars: float = 0,
    reviews: int = 0,
    bought_in_last_month: int = 0,
    is_best_seller: bool = False,
    cluster_id: int,
):

    raw_product = pd.DataFrame(
        [
            {
                "title": title,

                "category_name":
                    category_name,

                "stars":
                    stars,

                "reviews":
                    reviews,

                "boughtInLastMonth":
                    bought_in_last_month,

                "isBestSeller":
                    is_best_seller,

                "cluster_id":
                    cluster_id,
            }
        ]
    )


    # EXACT SAME FEATURE ENGINEERING
    processed_product = (
        create_features(
            raw_product
        )
    )


    structured_features = (
        production_preprocessor
        .transform(
            processed_product
        )
    )


    title_features = (
        production_tfidf
        .transform(
            processed_product[
                "title"
            ]
        )
    )


    final_features = hstack(
        [
            csr_matrix(
                structured_features
            ),

            title_features,
        ],
        format="csr",
    )


    log_prediction = (
        production_model
        .predict(
            final_features
        )[0]
    )


    predicted_price = float(
        np.expm1(
            log_prediction
        )
    )


    predicted_price = max(
        0.0,
        predicted_price,
    )


    return {

        "predicted_price":
            round(
                predicted_price,
                2,
            ),

        "model_version":
            "production_v1",
    }

In [3]:
result = predict_product_price(

    title=(
        "Dell Latitude Laptop "
        "Intel Core i7 16GB RAM "
        "512GB SSD 15.6 Inch"
    ),

    category_name=(
        "Computers & Tablets"
    ),

    stars=4.5,

    reviews=850,

    bought_in_last_month=100,

    is_best_seller=False,

    # Temporary.
    # We automate this in the next stage.
    cluster_id=31,
)

print(result)

{'predicted_price': 479.74, 'model_version': 'production_v1'}


In [4]:
from pathlib import Path

CURRENT_DIRECTORY = Path.cwd()

PROJECT_ROOT = (
    CURRENT_DIRECTORY.parent
    if CURRENT_DIRECTORY.name.lower() == "notebooks"
    else CURRENT_DIRECTORY
)

CLUSTER_MODEL_ROOT = (
    PROJECT_ROOT
    / "models"
    / "clustering"
)

files = [
    "incremental_pca.joblib",
    "structured_scaler.joblib",
    "minibatch_kmeans.joblib",
]

print("=" * 80)
print("CLUSTERING MODEL CHECK")
print("=" * 80)

for filename in files:

    path = (
        CLUSTER_MODEL_ROOT
        / filename
    )

    print(
        filename,
        "→",
        path.exists(),
        path,
    )

CLUSTERING MODEL CHECK
incremental_pca.joblib → True /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/models/clustering/incremental_pca.joblib
structured_scaler.joblib → True /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/models/clustering/structured_scaler.joblib
minibatch_kmeans.joblib → True /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/models/clustering/minibatch_kmeans.joblib


In [5]:
import joblib
from pathlib import Path


# ============================================================
# PATHS
# ============================================================

CURRENT_DIRECTORY = Path.cwd()

PROJECT_ROOT = (
    CURRENT_DIRECTORY.parent
    if CURRENT_DIRECTORY.name.lower() == "notebooks"
    else CURRENT_DIRECTORY
)

CLUSTER_MODEL_ROOT = (
    PROJECT_ROOT
    / "models"
    / "clustering"
)

PCA_FILE = (
    CLUSTER_MODEL_ROOT
    / "incremental_pca.joblib"
)

SCALER_FILE = (
    CLUSTER_MODEL_ROOT
    / "structured_scaler.joblib"
)

KMEANS_FILE = (
    CLUSTER_MODEL_ROOT
    / "minibatch_kmeans.joblib"
)


# ============================================================
# LOAD MODELS
# ============================================================

pca = joblib.load(PCA_FILE)
scaler = joblib.load(SCALER_FILE)
kmeans = joblib.load(KMEANS_FILE)


# ============================================================
# INSPECT
# ============================================================

print("=" * 80)
print("CLUSTERING MODEL INSPECTION")
print("=" * 80)

print("\nPCA")
print("-" * 50)

print(
    "Input features:",
    getattr(
        pca,
        "n_features_in_",
        "unknown",
    )
)

print(
    "Output components:",
    getattr(
        pca,
        "n_components_",
        getattr(
            pca,
            "n_components",
            "unknown",
        ),
    )
)


print("\nSTRUCTURED SCALER")
print("-" * 50)

print(
    "Expected features:",
    getattr(
        scaler,
        "n_features_in_",
        "unknown",
    )
)

if hasattr(
    scaler,
    "feature_names_in_",
):
    print(
        "Feature names:",
        list(
            scaler.feature_names_in_
        )
    )


print("\nKMEANS")
print("-" * 50)

print(
    "Expected input features:",
    getattr(
        kmeans,
        "n_features_in_",
        "unknown",
    )
)

print(
    "Number of clusters:",
    getattr(
        kmeans,
        "n_clusters",
        "unknown",
    )
)

print(
    "Cluster center shape:",
    kmeans.cluster_centers_.shape,
)

CLUSTERING MODEL INSPECTION

PCA
--------------------------------------------------
Input features: 384
Output components: 64

STRUCTURED SCALER
--------------------------------------------------
Expected features: 6
Feature names: ['price_log1p', 'list_price_log1p', 'stars', 'reviews_log1p', 'bought_log1p', 'is_best_seller']

KMEANS
--------------------------------------------------
Expected input features: 70
Number of clusters: 100
Cluster center shape: (100, 70)


In [6]:
from pathlib import Path
import numpy as np
import pandas as pd


CURRENT_DIRECTORY = Path.cwd()

PROJECT_ROOT = (
    CURRENT_DIRECTORY.parent
    if CURRENT_DIRECTORY.name.lower() == "notebooks"
    else CURRENT_DIRECTORY
)

FEATURE_ROOT = (
    PROJECT_ROOT
    / "data"
    / "clustering"
    / "amazon"
    / "features"
)

print("=" * 80)
print("CLUSTER FEATURE FILES")
print("=" * 80)

for path in sorted(FEATURE_ROOT.glob("*")):

    if path.is_file():

        size_mb = (
            path.stat().st_size
            / 1024
            / 1024
        )

        print(
            f"{path.name:45s} "
            f"{size_mb:10.2f} MB"
        )

CLUSTER FEATURE FILES
amazon_cluster_features.npy                       372.12 MB
amazon_structured_features.npy                     31.90 MB
amazon_text_embedding_index.parquet               109.83 MB
amazon_text_embeddings.npy                       2041.35 MB
amazon_text_pca.npy                               340.23 MB


In [7]:
files_to_check = [
    "amazon_text_embeddings.npy",
    "amazon_text_pca.npy",
    "amazon_structured_features.npy",
    "amazon_cluster_features.npy",
]

print()
print("=" * 80)
print("ARRAY SHAPES")
print("=" * 80)

for filename in files_to_check:

    path = (
        FEATURE_ROOT
        / filename
    )

    if path.exists():

        array = np.load(
            path,
            mmap_mode="r",
        )

        print(
            filename,
            "→",
            array.shape,
            array.dtype,
        )

    else:

        print(
            filename,
            "→ MISSING"
        )


ARRAY SHAPES
amazon_text_embeddings.npy → (1393564, 384) float32
amazon_text_pca.npy → (1393564, 64) float32
amazon_structured_features.npy → (1393564, 6) float32
amazon_cluster_features.npy → (1393564, 70) float32


In [8]:
import pandas as pd


INDEX_FILE = (
    FEATURE_ROOT
    / "amazon_text_embedding_index.parquet"
)

index_df = pd.read_parquet(
    INDEX_FILE
)

print("=" * 80)
print("TEXT EMBEDDING INDEX")
print("=" * 80)

print("Rows:", f"{len(index_df):,}")

print("\nColumns:")
print(index_df.columns.tolist())

print("\nFirst 10 rows:")
display(index_df.head(10))

print("\nMissing ASIN:")
if "asin" in index_df.columns:
    print(index_df["asin"].isna().sum())

print("\nDuplicate ASIN:")
if "asin" in index_df.columns:
    print(index_df["asin"].duplicated().sum())

TEXT EMBEDDING INDEX
Rows: 1,393,564

Columns:
['asin', 'title', 'category_name', 'price', 'price_band', 'embedding_row']

First 10 rows:


,asin,title,category_name,price,price_band,embedding_row
0,B014TMV5YE,"Sion Softside Expandable Roller Luggage, Black...",Suitcases,139.99,premium,0
1,B07GDLCQXV,Luggage Sets Expandable PC+ABS Durable Suitcas...,Suitcases,169.99,luxury,1
2,B07XSCCZYG,Platinum Elite Softside Expandable Checked Lug...,Suitcases,365.49,luxury,2
3,B08MVFKGJM,Freeform Hardside Expandable with Double Spinn...,Suitcases,291.59,luxury,3
4,B01DJLKZBA,Winfield 2 Hardside Expandable Luggage with Sp...,Suitcases,174.99,luxury,4
5,B07XSCD2R4,Maxlite 5 Softside Expandable Luggage with 4 S...,Suitcases,144.49,premium,5
6,B07MXF4G8K,"Hard Shell Carry on Luggage Airline Approved, ...",Suitcases,169.99,luxury,6
7,B07H515VCZ,"Maxporter II 30"" Hardside Spinner Trunk Luggag...",Suitcases,299.99,luxury,7
8,B08BXBCNMQ,Omni 2 Hardside Expandable Luggage with Spinne...,Suitcases,112.63,premium,8
9,B0B9K44XTS,Luggage Sets Expandable Lightweight Suitcases ...,Suitcases,209.99,luxury,9



Missing ASIN:
0

Duplicate ASIN:
0


In [9]:
from pathlib import Path
import pandas as pd


CURRENT_DIRECTORY = Path.cwd()

PROJECT_ROOT = (
    CURRENT_DIRECTORY.parent
    if CURRENT_DIRECTORY.name.lower() == "notebooks"
    else CURRENT_DIRECTORY
)

INDEX_FILE = (
    PROJECT_ROOT
    / "data"
    / "clustering"
    / "amazon"
    / "features"
    / "amazon_text_embedding_index.parquet"
)

AMAZON_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "amazon"
    / "amazon_price_training.parquet"
)


# ============================================================
# LOAD ONLY ASIN
# ============================================================

index_df = pd.read_parquet(
    INDEX_FILE,
    columns=[
        "asin",
        "embedding_row",
    ],
)

amazon_df = pd.read_parquet(
    AMAZON_FILE,
    columns=[
        "asin",
    ],
)


# ============================================================
# CHECK
# ============================================================

index_asin = (
    index_df["asin"]
    .astype(str)
    .reset_index(drop=True)
)

amazon_asin = (
    amazon_df["asin"]
    .astype(str)
    .reset_index(drop=True)
)

print("=" * 80)
print("FINAL ROW ALIGNMENT CHECK")
print("=" * 80)

print(
    "Index rows:",
    f"{len(index_asin):,}"
)

print(
    "Amazon rows:",
    f"{len(amazon_asin):,}"
)

same_count = (
    len(index_asin)
    == len(amazon_asin)
)

same_order = (
    same_count
    and index_asin.equals(
        amazon_asin
    )
)

embedding_row_ok = (
    index_df["embedding_row"]
    .reset_index(drop=True)
    .equals(
        pd.Series(
            range(len(index_df)),
            name="embedding_row",
        )
    )
)

print()
print(
    "Same row count:",
    same_count
)

print(
    "Exact ASIN order:",
    same_order
)

print(
    "Embedding row sequence valid:",
    embedding_row_ok
)


# ============================================================
# FIND FIRST MISMATCH IF ANY
# ============================================================

if same_count and not same_order:

    mismatch_mask = (
        index_asin
        != amazon_asin
    )

    first_mismatch = (
        mismatch_mask.idxmax()
    )

    print()
    print(
        "❌ First mismatch at row:",
        first_mismatch
    )

    print(
        "Index ASIN:",
        index_asin.iloc[
            first_mismatch
        ]
    )

    print(
        "Amazon ASIN:",
        amazon_asin.iloc[
            first_mismatch
        ]
    )


# ============================================================
# FINAL DECISION
# ============================================================

if (
    same_count
    and same_order
    and embedding_row_ok
):

    print()
    print(
        "✅ ROW ALIGNMENT VERIFIED"
    )

    print(
        "Safe to rebuild clean clustering."
    )

else:

    print()
    print(
        "❌ ALIGNMENT FAILED"
    )

    print(
        "Do NOT train clean KMeans yet."
    )

FINAL ROW ALIGNMENT CHECK
Index rows: 1,393,564
Amazon rows: 1,393,564

Same row count: True
Exact ASIN order: True
Embedding row sequence valid: True

✅ ROW ALIGNMENT VERIFIED
Safe to rebuild clean clustering.
